# Fine-tuning an Open-source Transformer for Financial Sentiment Classification

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/helenlu-vbs/NLP_LLM_for_Finance_and-Accounting_Research-Sheffield-/blob/main/004_finetune_open_source_transformer_financial_sentiment.ipynb)

This notebook is a Google Colab teaching demo for a finance/accounting NLP workshop.

We fine-tune an open-source transformer model on labelled financial sentences from the Hugging Face dataset `atrost/financial_phrasebank`. The task is three-class sentiment classification:

- `negative`
- `neutral`
- `positive`

The default model is `distilbert/distilbert-base-uncased`, which is small enough for a live classroom demo. An optional finance-specific backbone, `yiyanghkust/finbert-pretrain`, is included later but is not used by default.

## Learning Objectives

By the end of this notebook, students should understand:

1. What fine-tuning means.
2. How labelled text data are used to adapt a general language model to a finance-specific task.
3. The difference between a pretrained model and a fine-tuned classifier.
4. How to evaluate a fine-tuned model using accuracy, macro F1, and a confusion matrix.
5. Why fine-tuning is useful for repeated classification tasks but less suitable for complex reasoning/extraction tasks.

## Colab Data Requirements

This notebook does **not** require any raw CSV files, local folders, Google Drive mounts, or API keys. When opened from GitHub in Colab, it downloads the labelled dataset from Hugging Face (`atrost/financial_phrasebank`) and downloads the selected model weights from Hugging Face.


## 1. Setup

This notebook is designed for Google Colab. It installs the required packages, imports the main libraries, and fixes random seeds for reproducibility.

Packages used:

- `transformers`
- `datasets`
- `evaluate`
- `accelerate`
- `scikit-learn`
- `torch`

In [ ]:
!pip -q install transformers datasets evaluate accelerate scikit-learn

In [ ]:
import os
import random
import inspect

import numpy as np
import pandas as pd
import torch

from datasets import load_dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    pipeline,
)

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

os.environ["WANDB_DISABLED"] = "true"

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Explain the Model

We use `distilbert/distilbert-base-uncased` as the default open-source model.

DistilBERT is a smaller, faster version of BERT. It is suitable for a live classroom demo because it trains more quickly than full BERT while still illustrating the transformer fine-tuning workflow.

Optional finance-specific backbone:

```python
MODEL_NAME = "yiyanghkust/finbert-pretrain"
```

`yiyanghkust/finbert-pretrain` is pretrained on financial communication text, including 10-K/10-Q filings, earnings calls, and analyst reports. It may be more finance-aware, but we keep DistilBERT as the default for speed and reliability in class.

## 3. Load Data

We use the Hugging Face dataset `atrost/financial_phrasebank`.

The dataset has financial sentences labelled as negative, neutral, or positive. Different dataset configurations/splits may expose labels as strings or integers, so the code below is written defensively.

In [ ]:
DATASET_NAME = "atrost/financial_phrasebank"

raw = load_dataset(DATASET_NAME)
print(raw)
print("Available splits:", list(raw.keys()))

In [ ]:
# Some Hugging Face datasets provide train/validation/test splits.
# If a dataset only provides one split, create train/validation/test splits.
if {"train", "validation", "test"}.issubset(set(raw.keys())):
    dataset = DatasetDict({
        "train": raw["train"],
        "validation": raw["validation"],
        "test": raw["test"],
    })
else:
    first_split = raw[list(raw.keys())[0]]
    split_1 = first_split.train_test_split(test_size=0.20, seed=SEED, stratify_by_column="label" if "label" in first_split.column_names else None)
    split_2 = split_1["test"].train_test_split(test_size=0.50, seed=SEED, stratify_by_column="label" if "label" in split_1["test"].column_names else None)
    dataset = DatasetDict({
        "train": split_1["train"],
        "validation": split_2["train"],
        "test": split_2["test"],
    })

print(dataset)
for split in ["train", "validation", "test"]:
    print(f"\nExample from {split}:")
    print(dataset[split][0])

In [ ]:
label2id = {"negative": 0, "neutral": 1, "positive": 2}
id2label = {0: "negative", 1: "neutral", 2: "positive"}

# Detect text and label columns.
possible_text_cols = ["sentence", "text", "content"]
TEXT_COL = next((c for c in possible_text_cols if c in dataset["train"].column_names), None)
if TEXT_COL is None:
    raise ValueError(f"Could not find text column. Columns are: {dataset['train'].column_names}")

possible_label_cols = ["label", "labels", "sentiment"]
LABEL_COL = next((c for c in possible_label_cols if c in dataset["train"].column_names), None)
if LABEL_COL is None:
    raise ValueError(f"Could not find label column. Columns are: {dataset['train'].column_names}")

print("Text column:", TEXT_COL)
print("Original label column:", LABEL_COL)

# Robust label mapping: labels may be integers, ClassLabel objects, or strings.
label_feature = dataset["train"].features.get(LABEL_COL)
class_names = getattr(label_feature, "names", None)
print("Dataset class names:", class_names)

# Handle possible class-name ordering differences defensively.
def normalize_label(example):
    value = example[LABEL_COL]
    if isinstance(value, str):
        label_name = value.strip().lower()
    elif class_names is not None:
        label_name = class_names[int(value)].strip().lower()
    else:
        # Most Financial PhraseBank versions use 0=negative, 1=neutral, 2=positive.
        label_name = id2label[int(value)]
    example["labels"] = label2id[label_name]
    return example

dataset = dataset.map(normalize_label)

# Keep the original text column name, but create a consistent "sentence" column if needed.
if TEXT_COL != "sentence":
    dataset = dataset.rename_column(TEXT_COL, "sentence")
    TEXT_COL = "sentence"

print(dataset)
print("Final columns:", dataset["train"].column_names)

In [ ]:
# Quick pandas table for label counts.
train_df = dataset["train"].to_pandas()
train_df["label_name"] = train_df["labels"].map(id2label)
train_df["label_name"].value_counts().rename_axis("label").reset_index(name="count")

## 4. Quick Data Inspection

The input is one financial sentence.

The output is one of three sentiment labels: negative, neutral, or positive.

This is supervised classification, not text generation. The model learns from labelled examples.

In [ ]:
for split in ["train", "validation", "test"]:
    df = dataset[split].to_pandas()
    df["label_name"] = df["labels"].map(id2label)
    print(f"\n{split.upper()} class balance")
    print(df["label_name"].value_counts().sort_index())

print("\nFive random training examples:")
sample_df = train_df.sample(5, random_state=SEED)[["sentence", "label_name"]]
display(sample_df)

## 5. Load Tokenizer and Model

We load the tokenizer and a sequence-classification model with three output labels.

The base language model is pretrained. The classification head is task-specific and will be adapted during fine-tuning.

In [ ]:
MODEL_NAME = "distilbert/distilbert-base-uncased"
# Optional finance-specific backbone, slower but finance-aware:
# MODEL_NAME = "yiyanghkust/finbert-pretrain"

MAX_LENGTH = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=id2label,
    label2id=label2id,
)

print("Loaded model:", MODEL_NAME)

## 6. Tokenize Data

Transformers do not consume raw strings directly. We tokenize each sentence into token IDs and attention masks.

Padding is handled dynamically by `DataCollatorWithPadding`.

In [ ]:
def tokenize_function(batch):
    return tokenizer(batch["sentence"], truncation=True, max_length=MAX_LENGTH)

tokenized_dataset = dataset.map(tokenize_function, batched=True)

# Remove columns that Trainer does not need. Keep labels and tokenized fields.
keep_cols = {"input_ids", "attention_mask", "labels"}
for split in tokenized_dataset:
    remove_cols = [c for c in tokenized_dataset[split].column_names if c not in keep_cols]
    tokenized_dataset[split] = tokenized_dataset[split].remove_columns(remove_cols)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
print(tokenized_dataset)

## 7. Baseline Before Fine-tuning

Before fine-tuning, the transformer body is pretrained, but the classification head has not yet learned this three-class financial sentiment task.

Therefore, predictions from the newly initialized classification head are not meaningful yet.

In [ ]:
def predict_examples(model, tokenizer, sentences, device=None):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()
    encoded = tokenizer(sentences, padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors="pt")
    encoded = {k: v.to(device) for k, v in encoded.items()}
    with torch.no_grad():
        logits = model(**encoded).logits
    probs = torch.softmax(logits, dim=-1).cpu().numpy()
    pred_ids = probs.argmax(axis=1)
    return pd.DataFrame({
        "sentence": sentences,
        "prediction": [id2label[int(i)] for i in pred_ids],
        "prob_negative": probs[:, 0],
        "prob_neutral": probs[:, 1],
        "prob_positive": probs[:, 2],
    })

baseline_sentences = [dataset["test"][i]["sentence"] for i in range(5)]
predict_examples(model, tokenizer, baseline_sentences)

## 8. Fine-tune Model

Fine-tuning updates the model parameters using labelled examples from the financial sentiment dataset.

We train for only two epochs to keep the runtime manageable for a live Colab demo.

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
    }

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

base_training_args = dict(
    output_dir="./distilbert_financial_phrasebank",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=2,
    weight_decay=0.01,
    save_strategy="epoch",
    logging_steps=20,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    report_to="none",
    seed=SEED,
)

# Transformers changed the argument name from evaluation_strategy to eval_strategy.
# Try the newer name first; fall back for older versions.
try:
    training_args = TrainingArguments(eval_strategy="epoch", **base_training_args)
except TypeError:
    training_args = TrainingArguments(evaluation_strategy="epoch", **base_training_args)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

## 9. Evaluate on Held-out Test Set

The test set was not used for training. It gives us a cleaner estimate of how the fine-tuned classifier performs on unseen labelled data.

Macro F1 is useful when class sizes differ because it gives equal weight to each class rather than letting the largest class dominate the score.

In [ ]:
test_metrics = trainer.evaluate(tokenized_dataset["test"])
print(test_metrics)

pred_output = trainer.predict(tokenized_dataset["test"])
y_true = pred_output.label_ids
y_pred = np.argmax(pred_output.predictions, axis=-1)

acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred, average="macro")
cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])

print("Accuracy:", round(acc, 4))
print("Macro F1:", round(f1, 4))
print("\nConfusion matrix, rows=true labels, columns=predicted labels")
print(pd.DataFrame(cm, index=["true_negative", "true_neutral", "true_positive"], columns=["pred_negative", "pred_neutral", "pred_positive"]))
print("\nClassification report")
print(classification_report(y_true, y_pred, target_names=["negative", "neutral", "positive"]))

## 10. Test on New Finance Examples

Now we classify new financial sentences using the fine-tuned model.

In [ ]:
new_examples = [
    "Operating profit decreased to EUR 11.2 million from EUR 16.6 million.",
    "EBIT margin was up from 1.4 percent to 5.1 percent.",
    "The company announced a new share repurchase programme.",
    "The firm warned that weak demand will reduce revenue next quarter.",
    "The acquisition is expected to close by the end of August.",
]

clf = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1,
    return_all_scores=True,
)

rows = []
for sentence, scores in zip(new_examples, clf(new_examples)):
    score_map = {s["label"].lower(): s["score"] for s in scores}
    pred_label = max(score_map, key=score_map.get)
    rows.append({
        "sentence": sentence,
        "prediction": pred_label,
        "prob_negative": score_map.get("negative", np.nan),
        "prob_neutral": score_map.get("neutral", np.nan),
        "prob_positive": score_map.get("positive", np.nan),
    })

pd.DataFrame(rows)

## 11. Save Model Locally

The fine-tuned model can be saved and reused later to classify new financial sentences.

This does not push anything to Hugging Face Hub and does not require login.

In [ ]:
SAVE_DIR = "./fine_tuned_financial_sentiment_model"
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print("Saved to:", SAVE_DIR)

reloaded_tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR)
reloaded_model = AutoModelForSequenceClassification.from_pretrained(SAVE_DIR)

predict_examples(reloaded_model, reloaded_tokenizer, [new_examples[0]])

## 12. Teaching Discussion

### What has the model learned?

It learned to map financial sentences to negative / neutral / positive labels.

### What does it not do?

It does not reason, extract causal mechanisms, or generate explanations. It is a classifier trained to predict one label per sentence.

### When is this useful?

Fine-tuning is useful for repeated classification at scale, especially when the task is well-defined and labelled examples exist.

### When is frontier LLM + RAG better?

A frontier LLM plus retrieval-augmented generation is often better when the task requires complex reasoning, structured extraction, or using context spread across multiple passages.

### Connection to Li et al. 2026

This fine-tuning demo teaches the classifier part of the NLP toolkit. Li et al. use a hybrid workflow: filtering/retrieval with traditional NLP or transformer tools, then frontier LLM extraction for culture type, tone, causes, effects, and causal triples.

## 13. Optional Extension: Use FinBERT-pretrain

Do not run this during the main demo unless you have extra time.

To switch to a finance-specific backbone, restart the runtime and change the model name near the top of the notebook:

```python
MODEL_NAME = "yiyanghkust/finbert-pretrain"
```

This model is finance-specific but may be slower than DistilBERT. It is useful for discussing how domain pretraining differs from task fine-tuning.

In [ ]:
# Optional extension only. Do not run by default.
# MODEL_NAME = "yiyanghkust/finbert-pretrain"
# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# model = AutoModelForSequenceClassification.from_pretrained(
#     MODEL_NAME,
#     num_labels=3,
#     id2label=id2label,
#     label2id=label2id,
# )